In [2]:
import torch
import torchaudio
import librosa
from pydub import AudioSegment
import numpy as np
import random
import soundfile as sf

In [3]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [4]:
audio_file = '../speech.wav'
audio, sr = librosa.load(audio_file, sr=None)

c:\Users\jiova\anaconda3\envs\speech_recognition\lib\site-packages\librosa\core\intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [ ]:
def predict_text(audio_file):
    waveform, sr = torchaudio.load(audio_file)

    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedCTCdecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor) -> str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return ''.join([self.labels[i] for i in indices])

    decoder = GreedCTCdecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript


In [6]:
def export_function(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file name must ends with .wav")
    audio.export(file_name, format="wav")
    print(f"audio saved as {file_name}")


def export_librosa(audio):
    while True:
        file_name = input("input filename: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file name must ends with .wav")
    sf.write(file_name, audio, sr)

In [7]:
def adjvolume(clip):
    while True:
        db_input = int(input("input db: "))
        if db_input > -1000 and db_input < 100:
            break
        else:
            print("must be between -1000 to 100")

    clip = clip + db_input
    export_function(clip)

def adjpitch():
    while True:
        pitch_rate = int(input("input pitch rate: "))
        if pitch_rate > -10 and pitch_rate < 10:
            break
        else:
            print("must be between -10 to 10")

    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)

def fade_in(clip):
    while True:
        duration = int(input("input duration: "))
        if duration > 1 and duration < 5:
            break
        else:
            print("must be between 1 to 5")
    # duration = random.randint(1,5)*1000

    clip = clip.fade_in(duration)
    export_function(clip)

def fade_out(clip):
    while True:
        duration = int(input("input duration: "))
        if duration > 1 and duration < 5:
            break
        else:
            print("must be between 1 to 5")
    # duration = random.randint(1,5)*1000

    clip = clip.fade_out(duration)
    export_function(clip)


In [8]:
def menu1():
    clip = AudioSegment.from_wav(audio_file)
    while True:
        print("1. increase/dec volume")
        print("2. increase/dec pitch")
        print("3. insert fadein")
        print("4. insert fadeout")
        print("5. back to main menu")

        choice = int(input(">> "))
        if choice == 1:
            adjvolume(clip)
        elif choice ==2:
            adjpitch()
        elif choice ==3:
            fade_in(clip)
        elif choice ==4:
            fade_out(clip)
        elif choice==5:
            break
        else:
            print("invalid input")


In [9]:
def mainmenu():
    while True:
        print("1. audio segmentation")
        print("2. predict text")
        print("3. exit")

        choice = int(input(">> "))
        if choice ==1:
            menu1()
        elif choice ==2:
            transcript = predict_text(audio_file)
            print(f"hasil transcript {transcript}")
        elif choice ==3:
            break
        else:
            print("invalid input")
    print("thx")

In [10]:
mainmenu()

1. audio segmentation
2. predict text
3. exit
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. back to main menu
audio saved as wav.wav
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. back to main menu
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. back to main menu
audio saved as wav.wav
1. increase/dec volume
2. increase/dec pitch
3. insert fadein
4. insert fadeout
5. back to main menu
1. audio segmentation
2. predict text
3. exit
hasil transcript I HAVE THAT CURIOSITY BESIDE ME AT THIS MOMENT 
1. audio segmentation
2. predict text
3. exit
thx
